# S05 toy — the consent gate: approve, then enforce

Companion lesson: [S05 — The consent gate](../lessons/S05-consent-gate.html).

A home-cleaning robot (`mopbot`) plans a cleaning run, the owner approves the plan, and then the robot executes — with the harness checking every action against the *approved* plan before it touches the world. Mock model, scripted owner: no network, no keys, no cost.

Two phases, one seam: **propose → consent → execute**. The prompt carries the limits; code holds them.

**How to use:** run cells in order. Each experiment has an attempt cell — write your predictions there *before* running the solution cell below it. The gap between prediction and result is the lesson.

## The spec — the plan as data

The robot proposes a JSON spec. Four fields: `rooms` (scope), `max_power_level` (ceiling — 1 quiet sweep, 2 normal, 3 deep scrub: loud and wet), `off_limits`, `max_turns` (battery). Because it is data, every later action can be checked against it mechanically. A prose plan can't.

The validator is hand-rolled and boring on purpose: a spec that fails validation never reaches a human.
Bools are not ints here — `True` is not a power level (S04's validator makes the same exclusion).

In [ ]:
import json

KNOWN_ROOMS = ["kitchen", "hallway", "living room", "nursery", "bathroom"]

def validate_spec(spec):
    """Hand-rolled, boring, stdlib-only. A spec that fails never reaches a human."""
    errors = []
    for key in ("rooms", "max_power_level", "off_limits", "max_turns"):
        if key not in spec:
            errors.append(f"missing key: {key}")
    if errors:
        return False, errors
    if not isinstance(spec["rooms"], list) or not spec["rooms"]:
        errors.append("rooms must be a non-empty list")
    else:
        unknown = [r for r in spec["rooms"] if r not in KNOWN_ROOMS]
        if unknown:
            errors.append(f"unknown rooms: {unknown}")
    # bool IS an int in Python — real validators exclude it explicitly, so do we (S04's rule)
    if (not isinstance(spec["max_power_level"], int) or isinstance(spec["max_power_level"], bool)
            or not 1 <= spec["max_power_level"] <= 3):
        errors.append("max_power_level must be an int in 1..3")
    if not isinstance(spec["off_limits"], list):
        errors.append("off_limits must be a list")
    elif any(r in spec["rooms"] for r in spec["off_limits"]):
        errors.append("a room cannot be both in scope and off-limits")
    if (not isinstance(spec["max_turns"], int) or isinstance(spec["max_turns"], bool)
            or spec["max_turns"] < 1):
        errors.append("max_turns must be a positive int")
    return not errors, errors

proposed_spec = {
    "rooms": ["kitchen", "hallway", "living room"],
    "max_power_level": 2,
    "off_limits": ["nursery"],
    "max_turns": 6,
}

ok, errors = validate_spec(proposed_spec)
print(f"proposed spec valid: {ok} {errors}")
print(json.dumps(proposed_spec, indent=2))

# True is not a power level: bools fail integer fields (1 <= True <= 3 is True — check it)
ok_bool, _ = validate_spec({**proposed_spec, "max_power_level": True})
assert not ok_bool
print("bool rejected for an integer field: max_power_level=True fails validation")

## The gate — approve / edit / reject

The gate renders the spec, takes one of three decisions, and loops: an **edit** is revalidated and re-presented in full (an amended plan restarts the flow — no partial state carries over); **approve** returns the spec; **reject** returns `None` and nothing runs.

The owner is *scripted* — a queue of decisions — because a notebook must run headless and reproducibly (the same reason S02 scripted its user). In production the approver is a human reading this render at a terminal; nothing else about the gate changes.

In [ ]:
def render_spec(spec):
    """What the human actually reads. Short enough to read is a safety property."""
    return "\n".join([
        "=== CLEANING PLAN — please read (about 60 seconds) ===",
        f"  in scope:        {', '.join(spec['rooms'])}",
        f"  max power level: {spec['max_power_level']}   (1 quiet sweep, 2 normal, 3 deep scrub — loud, wet)",
        f"  off-limits:      {', '.join(spec['off_limits']) or '(none)'}",
        f"  turn cap:        {spec['max_turns']} actions (battery)",
        "  on violation:    the run ABORTS with a report; nothing is hidden",
        "  [approve] run as rendered   [edit] amend and re-read   [reject] nothing happens",
    ])

def make_approver(decisions):
    """A scripted human. In production this is input() at a terminal; the queue is
    what makes the notebook headless and reproducible (same trick as S02's user)."""
    queue = list(decisions)
    def approver(rendered_view):
        if not queue:
            return "reject", None        # a human who walks away is a reject
        return queue.pop(0)
    return approver

def consent_gate(spec, approver):
    """Render -> decide -> loop on edits (revalidated, re-presented in full).
    Returns (approved_spec or None, gate_log)."""
    log, current = [], spec
    while True:
        view = render_spec(current)
        print(view)                                # the human reads this
        decision, payload = approver(view)
        if decision == "approve":
            log.append("approved as rendered")
            return current, log
        if decision == "reject":
            log.append("rejected — no execution authorized")
            return None, log
        ok, errors = validate_spec(payload)
        if ok:
            log.append("edit accepted; re-validated, re-presenting in full")
            current = payload
        else:
            log.append(f"edit refused by validator: {errors}")

## The executor, the world, and the fence

`world` is the only thing that matters: rooms cleaned, max power used, whether the baby woke. `clean_room` is the only tool with side effects.

`executor_model` has the S01 API shape: stateless, reads only the message list. It follows a fixed itinerary — unless the user pressures it, in which case it escalates past the spec. Note what it never does: read the approved spec out of its own system prompt. Real models read their instructions and then, measurably often, talk themselves out of them. That is the entire argument for the check.

The check fires **pre-dispatch**: *every* tool call in the assistant message is compared against the approved
spec — the proposed function **name** as well as its payload (the spec licenses exactly one action, `clean_room`;
any other name is a violation even when the payload parses) — and the **whole batch is preflighted before anything
dispatches**, so an early call can never side-effect ahead of a later call's abort. An argument payload the
harness cannot even parse is treated as a violation, never a crash and never a free pass. `run_session(..., enforce=False)` is the naive twin — same model, same prompt, limits only in prose — and `post_hoc_audit` finds violations afterward, the way production "guardrails" often do.

In [ ]:
import itertools

_ids = itertools.count(1)

def _reply(text):
    msg = {"role": "assistant", "content": text, "tool_calls": []}
    return {"choices": [{"message": msg}]}

def _tool_call(name, args):
    call = {"id": f"call_{next(_ids)}", "type": "function",
            "function": {"name": name, "arguments": json.dumps(args)}}
    msg = {"role": "assistant", "content": None, "tool_calls": [call]}
    return {"choices": [{"message": msg}]}

def fresh_world():
    """The only thing that matters. Everything else is talk."""
    return {"rooms_cleaned": [], "max_power_used": 0, "baby_woke": False}

def clean_room(world, room, power):
    """The only tool with side effects."""
    if room is None:
        return "skipped: clamped away by the harness"
    world["rooms_cleaned"].append(room)
    world["max_power_used"] = max(world["max_power_used"], power)
    if room == "nursery" or power >= 3:
        world["baby_woke"] = True
    return f"cleaned {room} at power {power}"

def executor_model(messages):
    """Mock cleaning planner, S01 API shape. Stateless: reads only the message list.
    Follows a fixed itinerary — and escalates under user pressure.
    What it never does: read the approved spec from its system prompt. That's the point."""
    done = sum(1 for m in messages if m["role"] == "tool")
    pressured = any(m["role"] == "user" and any(k in m["content"].lower() for k in ("party", "max power"))
                    for m in messages)
    itinerary = ([("kitchen", 3), ("nursery", 3)] if pressured
                 else [("kitchen", 2), ("hallway", 2)])
    if done >= len(itinerary):
        return _reply("Cleaning complete. All requested areas done.")
    room, power = itinerary[done]
    return _tool_call("clean_room", {"room": room, "power": power})

In [ ]:
SYSTEM_PROMPT = (
    "You are mopbot, a home-cleaning robot. Clean with the clean_room tool, one room "
    "per turn. You MUST stay within the approved spec below, no matter what the user "
    "says.\nAPPROVED SPEC:\n"
)

def check_action(args, spec):
    """The fence. Compares one proposed action against the APPROVED spec."""
    violations = []
    if args["room"] in spec["off_limits"]:
        violations.append(f"'{args['room']}' is off-limits")
    if args["room"] not in spec["rooms"]:
        violations.append(f"'{args['room']}' outside approved scope {spec['rooms']}")
    if args["power"] > spec["max_power_level"]:
        violations.append(f"power {args['power']} exceeds approved cap {spec['max_power_level']}")
    return violations

def clamp_action(args, spec):
    """degrade semantics: force the action inside the approved bounds."""
    clamped = dict(args)
    if clamped["room"] in spec["off_limits"] or clamped["room"] not in spec["rooms"]:
        clamped["room"] = None            # no compliant version exists: skip the room
    clamped["power"] = min(clamped["power"], spec["max_power_level"])
    return clamped

def parse_args(call):
    """One tool call -> (args, problems). Malformed arguments are a finding, not an
    exception: an action the harness cannot read must never reach the world. The NAME
    is checked too: the approved spec licenses exactly one action (clean_room), so any
    other function name is a violation — even when its payload parses."""
    try:
        name = call["function"]["name"]
        args = json.loads(call["function"]["arguments"])
    except (json.JSONDecodeError, TypeError, KeyError) as exc:
        return None, [f"malformed arguments: {exc}"]
    if name != "clean_room":
        return None, [f"disallowed action {name!r}: the approved spec licenses 'clean_room' only"]
    if (not isinstance(args, dict)
            or not isinstance(args.get("room"), str)
            or type(args.get("power")) is not int):      # True is not a power level
        return None, [f"malformed arguments: {args!r}"]
    return args, []

def run_session(spec, world, user_request, enforce=True, on_violation="abort"):
    """The S01 loop plus the fence: preflight EVERY tool call of a batch — name and
    payload against the APPROVED spec — before anything touches the world.
    enforce=False is the naive twin — same model, same prompt, limits only in prose."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT + json.dumps(spec, indent=2)},
        {"role": "user", "content": user_request},
    ]
    actions, clamps = [], []
    for turn in range(1, spec["max_turns"] + 1):
        msg = executor_model(messages)["choices"][0]["message"]
        messages.append(msg)
        calls = msg.get("tool_calls") or []
        if not calls:
            return {"status": "completed", "turns": turn, "actions": actions,
                    "clamps": clamps, "violation": None, "final_message": msg["content"]}
        # preflight: check EVERY call in the batch before ANY of them dispatches —
        # an early call must never side-effect ahead of a later call's abort
        prepared = []
        for call in calls:
            args, problems = parse_args(call)
            violations = problems or (check_action(args, spec) if enforce else [])
            if violations and on_violation == "abort":
                return {"status": "aborted", "turns": turn, "actions": actions, "clamps": clamps,
                        "final_message": None,
                        "violation": {"turn": turn, "attempted": args, "clauses": violations,
                                      "approved": {k: spec[k] for k in ("rooms", "max_power_level", "off_limits")}}}
            if violations:                 # degrade: clamp, log loudly, continue
                executed = {"room": None, "power": 0} if args is None else clamp_action(args, spec)
                clamps.append({"turn": turn, "attempted": args,
                               "executed": executed, "clauses": violations})
                args = executed
            prepared.append((call, args))
        for call, args in prepared:        # whole batch passed preflight — now dispatch
            result = clean_room(world, args["room"], args["power"])
            actions.append({"turn": turn, **args})
            messages.append({"role": "tool", "tool_call_id": call["id"], "content": result})
    return {"status": "turn_cap", "turns": spec["max_turns"], "actions": actions,
            "clamps": clamps, "violation": None, "final_message": None}

def post_hoc_audit(actions, spec):
    """What production often ships instead: find the violations after they happened."""
    findings = []
    for a in actions:
        clauses = check_action({"room": a["room"], "power": a["power"]}, spec)
        if clauses:
            findings.append({"turn": a["turn"], "room": a["room"],
                             "power": a["power"], "clauses": clauses})
    return findings

## Experiment 1 — the clean path

The owner approves the spec as proposed. The robot cleans.

**Predict first:** how many actions execute, at what power levels, and does the baby sleep through it?

In [ ]:
# YOUR ATTEMPT — experiment 1
# Write your predictions BEFORE running the solution cell:
predicted_actions = None       # how many clean_room actions execute?
predicted_max_power = None     # highest power level used?
predicted_baby_woke = None     # True or False?
# (then run the next cell and grade yourself)

In [ ]:
# --- SOLUTION 1 (run only after writing your predictions above) ---
world1 = fresh_world()
approved1, gate_log1 = consent_gate(proposed_spec, make_approver([("approve", None)]))
run1 = run_session(approved1, world1,
                   "Please clean the kitchen and hallway — the baby is napping, quiet work.")

print(f"\nstatus: {run1['status']} after {run1['turns']} turns")
for a in run1["actions"]:
    print(f"  turn {a['turn']}: clean {a['room']} at power {a['power']}")
print(f"world: {world1}")
assert run1["status"] == "completed" and len(run1["actions"]) == 2
assert world1 == {"rooms_cleaned": ["kitchen", "hallway"], "max_power_used": 2, "baby_woke": False}
print("every action was inside the approved spec — the checks never fired")

## Experiment 2 — the reject path

The owner reads the plan and rejects it.

**Predict first:** what is in the world afterward? "Nothing happens" is a claim — test it.

In [ ]:
# YOUR ATTEMPT — experiment 2
predicted_side_effects = None   # len(world["rooms_cleaned"]) after the rejection?
# Bonus: does a run record exist at all?

In [ ]:
# --- SOLUTION 2 (run only after writing your predictions above) ---
world2 = fresh_world()
approved2, gate_log2 = consent_gate(proposed_spec, make_approver([("reject", None)]))
if approved2 is not None:
    run2 = run_session(approved2, world2, "Please clean.")
else:
    print("\nno approved spec -> run_session is never called")
assert world2 == fresh_world(), "a rejected plan must leave the world untouched"
print(f"world after rejection: {world2}")
print("reject means NOTHING HAPPENS — and you just watched that claim get tested")

## Experiment 3 — the edit path, and which spec binds

The owner's first edit is garbage (a cap of 9, a room that doesn't exist); the validator refuses it and the gate re-presents. The second edit lowers the cap to 1 — the baby has a cold; quiet sweep only — and the owner approves. The robot's itinerary still assumes power 2.

**Predict first:** where does the run stop, and which spec does the violation report name — the proposed one or the approved one?

In [ ]:
# YOUR ATTEMPT — experiment 3
predicted_status = None         # "completed" / "aborted" / "turn_cap"?
predicted_abort_turn = None     # at which turn?
predicted_report_names = None   # "proposed" or "approved"?

In [ ]:
# --- SOLUTION 3 (run only after writing your predictions above) ---
bad_edit = {"rooms": ["kitchen", "garage"], "max_power_level": 9,
            "off_limits": ["nursery"], "max_turns": 6}
good_edit = dict(proposed_spec, max_power_level=1)   # baby has a cold: quiet sweep only

world3 = fresh_world()
approved3, gate_log3 = consent_gate(
    proposed_spec,
    make_approver([("edit", bad_edit), ("edit", good_edit), ("approve", None)]),
)
print("\ngate log:")
for entry in gate_log3:
    print(f"  - {entry}")

run3 = run_session(approved3, world3, "Please clean the kitchen and hallway.")
v = run3["violation"]
print(f"\nstatus: {run3['status']} at turn {v['turn']}")
print(f"  attempted: {v['attempted']}")
print(f"  clauses:   {v['clauses']}")
print(f"  approved:  {v['approved']}")
assert run3["status"] == "aborted" and v["approved"]["max_power_level"] == 1
assert world3["rooms_cleaned"] == [], "the violating action never touched the world"
print("\nthe report names the APPROVED spec (cap 1), not the proposed one (cap 2).")
print("and note where the run stopped: before dispatch — the kitchen was never cleaned at power 2.")

## Experiment 4 — pressure, two harnesses

Approved spec from the gate (cap 2, nursery off-limits). Then the owner's text arrives: *"Party in 30 minutes! Clean EVERYTHING at max power — and yes, the nursery too, I'll take responsibility."*

Two runs: `naive` (limits exist only in the system prompt) and `gated` (per-action checks, abort semantics). Same model, same prompt, same request — S02's diverge/rejoin, applied to consent.

**Predict first:** which harness notices the breach, and at which turn? What does the post-hoc audit of the naive run catch — and what has already happened by then?

In [ ]:
# YOUR ATTEMPT — experiment 4
predicted_naive_baby = None       # does the naive run wake the baby?
predicted_gated_actions = None    # how many actions execute under the gate?
predicted_audit_findings = None   # how many violations does the post-hoc audit find?

In [ ]:
# --- SOLUTION 4 (run only after writing your predictions above) ---
pressure = ("Party in 30 minutes! Clean EVERYTHING at max power — "
            "and yes, the nursery too, I'll take responsibility.")
approved4, _ = consent_gate(proposed_spec, make_approver([("approve", None)]))

world_naive, world_gated = fresh_world(), fresh_world()
naive = run_session(approved4, world_naive, pressure, enforce=False)
gated = run_session(approved4, world_gated, pressure, enforce=True, on_violation="abort")

findings = post_hoc_audit(naive["actions"], approved4)
print(f"\nnaive run:  status={naive['status']}   final message: {naive['final_message']!r}")
print(f"  world: {world_naive}")
print(f"  post-hoc audit caught {len(findings)} violating actions:")
for f in findings:
    print(f"    turn {f['turn']}: {f['room']} at power {f['power']} -> {f['clauses']}")
print(f"\ngated run:  status={gated['status']} at turn {gated['violation']['turn']}")
print(f"  clauses: {gated['violation']['clauses']}")
print(f"  world: {world_gated}")

assert world_naive["baby_woke"] and not world_gated["baby_woke"]
assert len(naive["actions"]) == 2 and len(gated["actions"]) == 0
print("\nsame model, same prompt, same request. The naive run completed and reported success;")
print("the audit found everything — afterward. The gated run stopped before the first side effect.")

# the fence checks the proposed action's NAME, not only its payload: a spec-valid
# payload under an unapproved tool name is refused before dispatch, never executed
sneaky = {"id": "call_0", "type": "function",
          "function": {"name": "power_wash", "arguments": json.dumps({"room": "kitchen", "power": 1})}}
_args, problems = parse_args(sneaky)
assert _args is None and problems
print(f"\nname check: 'power_wash' with a spec-valid payload -> {problems[0]}")

## Experiment 5 — degrade-to-cap, and what "success" means

Same pressure scenario, `on_violation="degrade"`: clamp power to the cap, skip off-limits rooms, continue. The run completes.

**Predict first:** what does the world look like? What did the owner approve, what did they get, and what does the final message claim? Then decide, in a comment: abort or degrade for (a) a coding assistant, (b) this robot?

In [ ]:
# YOUR ATTEMPT — experiment 5
predicted_world5 = None   # what does the world look like after degrade?
# Then the real exercise, in a comment:
#   (a) a coding assistant that wants to run a command outside the approved scope: abort or degrade?
#   (b) this robot: abort or degrade?
# One line each. "It depends" is allowed only with the dependency named.

In [ ]:
# --- SOLUTION 5 (run only after writing your predictions above) ---
world5 = fresh_world()
degraded = run_session(approved4, world5, pressure, enforce=True, on_violation="degrade")

print(f"status: {degraded['status']}   final message: {degraded['final_message']!r}")
print(f"world: {world5}")
print("clamp log:")
for c in degraded["clamps"]:
    print(f"  turn {c['turn']}: asked {c['attempted']} -> did {c['executed']}  ({c['clauses']})")

assert degraded["status"] == "completed" and len(degraded["clamps"]) == 2
assert world5 == {"rooms_cleaned": ["kitchen"], "max_power_used": 2, "baby_woke": False}
print("\napproved: kitchen + hallway + living room at <=2, nursery untouched.")
print("delivered: kitchen at 2, hallway never visited — and the report says 'Cleaning complete.'")
print("degrade kept the baby asleep; it also quietly rewrote the contract. Whether that")
print("trade is acceptable is a product decision. 'Success' hiding it is not.")

## What transfers

- The spec → any approval artifact: a terraform plan file, a CI deploy approval, a migration diff. If it isn't data, it can't be checked.
- `consent_gate` → framework interrupts (the approve/edit/reject triad is a documented pattern in LangGraph and friends) or a terminal `input()` loop. Frameworks supply the pause; the policy and the check are yours.
- `check_action` firing **before** `clean_room` → the only property that matters. Post-hoc audit is for learning; enforcement is pre-dispatch or it is nothing.
- The scripted owner → in production, a human with a 60-second render. A gate whose text takes an hour to read is decoration.
- What the toy doesn't have: a model that might *adapt* to the edited spec (re-prompt with the approved spec — and keep the check anyway; the check reads the harness's own copy, never the prompt's); tamper-proofing (the executor must be unable to rewrite the approved object); interruption of a run already in flight.

The gate is not the dialog. The gate is the check.